# 04 · Design Storm (Alternating Blocks) + HEC-HMS Event Simulation

**Author:** Salvador Navas  
**Basin:** Río Besaya — Cantabria  
**Model:** Proyecto_Hec_HMS / Modelo_Besaya

## Classical methodology
1. Build design hyetographs (alternating blocks method) from the IDF curves of Notebook 03
2. Write `.gage` and `.met` files for HEC-HMS
3. Run HEC-HMS for each return period when the external binary is available
4. Read output hydrographs from DSS when available
5. Analyse peak flows and volumes

**Execution note.** The notebook always writes the HEC-HMS precipitation DSS with
`hecdss`. Launching the external HEC-HMS binary is disabled by default in
Azure/Jupyter sessions because Java/AWT can block kernels for several minutes.
Set `HYDRA_RUN_HEC_HMS=1` only in a validated container to run HMS. Otherwise,
the notebook falls back to a simplified SCS-CN + unit-hydrograph calculation.
Fallback peaks are useful for checking the data pipeline and monotonicity with
return period, but they are not calibrated HMS design discharges.


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, warnings
warnings.filterwarnings('ignore')

from pyhydra.modeling.hydrology.hec_hms import (
    read_gages, read_basin, read_subbasin, read_control,
    generate_gage, fill_gage_series,
    generate_met, generate_met_freq_storm,
    generate_control, generate_run, generate_py,
    run_hms_script, HMSModel,
)

# Resolve repo root: Jupyter sets cwd to the notebook dir;
# in Docker/script contexts cwd=/workspace (one level up)
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'data' / 'pilot_cases' / 'los_corrales_buelna').exists()),
    _cwd.parent.parent.parent if _cwd.name == 'los_corrales_buelna' else _cwd,
)
DATA_ROOT  = REPO_ROOT / 'data' / 'pilot_cases' / 'los_corrales_buelna'
PROC_DIR   = DATA_ROOT / 'processed'
HMS_DIR    = DATA_ROOT / 'models' / 'hec_hms'
OUT_DIR    = PROC_DIR / 'hms_events'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# HEC-HMS binary (Linux Docker) — ajustar si se ejecuta localmente con HEC-HMS instalado
HEC_HMS_BIN = Path('/workspace/data/hms/HEC-HMS-4.13/hec-hms.sh')

NAME_MODEL  = 'Project_1'
NAME_BASIN  = 'Modelo_Besaya'
PATH_MODEL  = str(HMS_DIR)

RETURN_PERIODS = [2, 5, 10, 25, 50, 100, 500]
STORM_DURATION = 24   # horas
TIME_STEP      = 60   # minutos
EPART          = '1HOUR' if TIME_STEP == 60 else f'{TIME_STEP}MIN'
FILE_DSS_DESIGN = 'Design_precip.dss'   # separate from Project_1.dss (DSS v6)

print(f'Modelo HMS: {PATH_MODEL}')
print(f'HEC-HMS binary: {HEC_HMS_BIN}  (existe: {HEC_HMS_BIN.exists()})')

Modelo HMS: /workspace/data/pilot_cases/los_corrales_buelna/models/hec_hms
HEC-HMS binary: /workspace/data/hms/HEC-HMS-4.13/hec-hms.sh  (existe: True)


---
## 1. Load IDF curves and GEV parameters

The IDF table and GEV parameters were exported by Notebook 03 as 
`extreme_value_params.json`. Key quantities used here:

- `idf[T][d]` — design intensity (mm/h) for return period T (years) and duration d (h)
- `gev_mu`, `gev_sigma`, `gev_xi` — for recomputing design depths in pyhydra
- `idf_temez_exponent` = 0.28 — Cantabrian climate scaling exponent

Pre-computed CC hyetograms (`Hietogramas/Cambio_Climatico/*.csv`) contain the 
scenario-scaled depths for RCP4.5 and RCP8.5 (10 CORDEX models each), already 
in alternating-block format. These are loaded here for comparison with the 
control period hyetograms.


In [2]:
HIET_CC_DIR = DATA_ROOT / 'gauges' / 'rainfall' / 'Hietogramas' / 'Cambio_Climatico'

# ── Hietogramas de control (sin CC) construidos desde IDF ──────────────────────
idf = pd.read_csv(PROC_DIR / 'idf_curves.csv')
with open(PROC_DIR / 'extreme_value_params.json') as f:
    ev_params = json.load(f)

print('Niveles de retorno 24h (hietograma de control):')
for T, v in ev_params['return_levels_24h'].items():
    print(f'  T={T:>4} años → {v:.1f} mm')

# ── Hietogramas CC pre-calculados (RCP4.5 y RCP8.5) ──────────────────────────
# Formato: index=tiempo (h), columnas=T_2, T_5, T_10, T_25, T_50, T_100, T_200, T_500
# Disponibles por subcuenca y por horizonte temporal

HORIZONTE = '2041_2070'   # cambiar a 2011_2040 o 2071_2100
CC_SCENARIOS = ['rcp45', 'rcp85']

cc_hietogramas = {}
for rcp in CC_SCENARIOS:
    # Usar subcuenca representativa (W1000 = cuenca principal)
    sample_sb = 'W1000'
    hiet_file = HIET_CC_DIR / f'Hietograma_bloques_{sample_sb}_{rcp}_{HORIZONTE}.csv'
    if hiet_file.exists():
        df = pd.read_csv(hiet_file, index_col=0)
        # Index comes as "0,063", "0,25", etc. (decimal hours with comma separator)
        df.index = df.index.astype(str).str.replace(',', '.').astype(float)
        cc_hietogramas[rcp] = df
        print(f'\n{rcp.upper()} {HORIZONTE}: {df.shape}  Periodos: {list(df.columns)}')
        print(df.head(3))
    else:
        print(f'  {hiet_file.name} no encontrado')

Niveles de retorno 24h (hietograma de control):
  T=   2 años → 48.4 mm
  T=   5 años → 58.2 mm
  T=  10 años → 64.8 mm
  T=  25 años → 73.5 mm
  T=  50 años → 80.1 mm
  T= 100 años → 86.8 mm
  T= 500 años → 102.8 mm

RCP45 2041_2070: (8, 8)  Periodos: ['T_2', 'T_5', 'T_10', 'T_25', 'T_50', 'T_100', 'T_200', 'T_500']
             T_2        T_5       T_10  ...      T_100      T_200      T_500
0.063   1.401602   2.163533   2.458819  ...   2.873241   2.922372   2.963865
0.250  13.251449  14.873423  15.502017  ...  16.384225  16.488812  16.577143
1.000  30.375519  33.535898  34.760700  ...  36.479662  36.683448  36.855558

[3 rows x 8 columns]

RCP85 2041_2070: (8, 8)  Periodos: ['T_2', 'T_5', 'T_10', 'T_25', 'T_50', 'T_100', 'T_200', 'T_500']
             T_2        T_5       T_10  ...      T_100      T_200      T_500
0.063   3.452116   4.214048   4.509334  ...   4.923756   4.972887   5.014380
0.250  12.411981  14.033955  14.662550  ...  15.544758  15.649345  15.737675
1.000  31.747800  

In [3]:
from pyhydra.climate.time_series.extremes import fit_gev, return_levels

# ── Demo pyhydra: recalcular GEV y niveles de retorno directamente ─────────────
# Load annual maxima (same data as Notebook 03)
DAT_FILES = sorted((DATA_ROOT / 'gauges' / 'rainfall').glob('Maximos_anual_*.dat'))
ann_max_series = {}
for dat in DAT_FILES:
    station = dat.stem.replace('Maximos_anual_', '')
    values  = np.loadtxt(dat)
    n       = len(values)
    years   = range(2012 - n + 1, 2013)
    idx     = pd.to_datetime([f'{y}-12-31' for y in years])
    ann_max_series[station] = pd.Series(values, index=idx, name=station)

annual_max = pd.DataFrame(ann_max_series).mean(axis=1).dropna()

params_gev = fit_gev(annual_max.values, method='mle')
print('GEV ajustado con pyhydra:')
print(f'  mu={params_gev["mu"]:.2f}  sigma={params_gev["sigma"]:.2f}  xi={params_gev["xi"]:.3f}')

rl_check = {T: round(float(return_levels(params_gev, [T], dist='gev').iloc[0]), 1)
            for T in [10, 25, 100, 500]}
print('\nNiveles de retorno P24h (mm):')
for T, v in rl_check.items():
    print(f'  T={T:>4} años → {v} mm')

GEV ajustado con pyhydra:
  mu=45.31  sigma=8.36  xi=0.032

Niveles de retorno P24h (mm):
  T=  10 años → 64.8 mm
  T=  25 años → 73.5 mm
  T= 100 años → 86.8 mm
  T= 500 años → 102.8 mm


---
## 2. Design hyetographs — Alternating blocks method

The alternating blocks method distributes rainfall increments from highest to lowest
intensity **symmetrically about the hyetograph centre**, producing the most 
unfavourable temporal pattern for runoff generation.

**Construction algorithm:**
1. For a duration D and time step Δt = 1h, compute the cumulative depths: 
   P(1h), P(2h), P(3h), … from the IDF curve
2. Compute increments: ΔP(i) = P(i·Δt) − P((i-1)·Δt)
3. Sort increments from largest to smallest
4. Arrange in alternating order: largest in the centre, then alternating left/right

**Why alternating blocks?**
- It is the Spanish design standard (CEDEX, Instrucción de Carreteras)
- It produces the maximum peak runoff for a given total depth (compared to uniform 
  or forward-sloping distributions)
- It is conservative: if the real storm pattern is less intense, the modelled 
  hydrograph peak is an upper bound

> **Limitation:** The alternating blocks pattern does not represent any single 
> observed storm. Real storms tend to have a forward-skewed pattern (peak near 
> the beginning). For calibration, use observed historical hyetographs from 
> Notebooks 05/06 instead of alternating blocks.


In [4]:
def alternating_blocks(idf_df: pd.DataFrame, T: int, duration_h: int,
                       dt_min: int = 60) -> pd.Series:
    """
    Genera hietograma por bloques alternos.

    Parameters
    ----------
    idf_df     : DataFrame con columnas T_years, duration_h, depth_mm
    T          : Periodo de retorno (años)
    duration_h : Duración total de la tormenta (h)
    dt_min     : Paso de tiempo (min)

    Returns
    -------
    Series con precipitación incremental por paso de tiempo (mm)
    """
    idf_T = idf_df[idf_df['T_years'] == T].set_index('duration_h')['depth_mm']
    dt_h  = dt_min / 60.0
    n_steps = int(duration_h / dt_h)
    durations = np.arange(dt_h, duration_h + dt_h, dt_h)

    # Depth interpolation at discrete durations
    depths = np.interp(durations, idf_T.index.values, idf_T.values)

    # Incrementos entre duraciones consecutivas
    increments = np.diff(np.concatenate([[0], depths]))

    # Ordenar en bloques alternos: mayor en el centro
    sorted_inc = np.sort(increments)[::-1]
    result = np.zeros(n_steps)
    lo, hi = n_steps // 2 - 1, n_steps // 2
    for k, val in enumerate(sorted_inc):
        if k % 2 == 0:
            result[hi] = val
            hi += 1
        else:
            result[lo] = val
            lo -= 1

    times = pd.date_range('2000-01-01', periods=n_steps, freq=f'{dt_min}min')
    return pd.Series(result, index=times, name=f'T{T}')


# Construir hietogramas para todos los T
hietogramas = {T: alternating_blocks(idf, T, STORM_DURATION, TIME_STEP)
               for T in RETURN_PERIODS}

# Visualisation
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
axes = axes.ravel()
for ax, T in zip(axes, RETURN_PERIODS):
    h = hietogramas[T]
    ax.bar(range(len(h)), h.values, color='steelblue', width=0.8)
    ax.set(title=f'T = {T} años  (P={h.sum():.1f} mm)',
           xlabel='Paso (h)', ylabel='mm/h')
axes[-1].set_visible(False)
plt.suptitle('Hietogramas de diseño — Bloques alternos — Cuenca Besaya', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'hietogramas_disenio.png', dpi=150)
plt.show()

In [5]:
# ── Hyetograph comparison: Control vs RCP4.5 vs RCP8.5 (T=100) ─────────────
T_COMPARE = 'T_100'
h_ctrl = hietogramas.get(100)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

if h_ctrl is not None:
    axes[0].bar(range(len(h_ctrl)), h_ctrl.values, color='steelblue', width=0.8)
    axes[0].set(title=f'Control T=100 años\n(P={h_ctrl.sum():.1f} mm)',
                xlabel='Paso (h)', ylabel='mm/h')

for ax, rcp, col in zip(axes[1:], CC_SCENARIOS, ['darkorange', 'tomato']):
    if rcp in cc_hietogramas and T_COMPARE in cc_hietogramas[rcp].columns:
        h = cc_hietogramas[rcp][T_COMPARE]
        # Total depth at 24h duration = last row (cumulative IDF depth)
        p_total = float(cc_hietogramas[rcp].loc[24.0, T_COMPARE])
        ax.bar(range(len(h)), h.values, color=col, width=0.8)
        label = rcp.upper().replace('RCP', 'RCP ')
        ax.set(title=f'{label} T=100 — {HORIZONTE}\n(P₂₄ₕ={p_total:.1f} mm)',
               xlabel='Paso (h)')
    else:
        ax.text(0.5, 0.5, f'{rcp} no disponible', ha='center', va='center',
                transform=ax.transAxes)

plt.suptitle(f'Hietogramas de diseño T=100 años — Control vs CC ({HORIZONTE})', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / f'hietogramas_CC_T100_{HORIZONTE}.png', dpi=150)
plt.show()

# ── Factores delta correctos: ratio de la profundidad a 24h (fila "24.0") ────────
# Los CC hietogramas son curvas IDF acumuladas (profundidad en mm) a distintas
# duraciones. El delta P_cc/P_ctrl se obtiene comparando la profundidad a 24h,
# no la suma de todas las duraciones.
DURATION_REF_H = 24.0

print(f'\nFactores delta P_CC/P_ctrl — profundidad a {DURATION_REF_H}h:')
rows = []
for rcp in CC_SCENARIOS:
    if rcp not in cc_hietogramas:
        continue
    df_rcp = cc_hietogramas[rcp]
    if DURATION_REF_H not in df_rcp.index:
        print(f'  {rcp}: fila {DURATION_REF_H}h no disponible — verificar formato del CSV')
        continue
    for Tcol in df_rcp.columns:
        T_yr = int(Tcol.replace('T_', ''))
        P_cc   = float(df_rcp.loc[DURATION_REF_H, Tcol])
        P_ctrl = ev_params['return_levels_24h'].get(str(T_yr), None)
        if P_ctrl is None or P_ctrl == 0:
            continue
        delta = round(P_cc / P_ctrl, 3)
        rows.append({'escenario': rcp, 'T_years': T_yr,
                     'P_ctrl_mm': round(P_ctrl, 1),
                     'P_cc_mm':   round(P_cc, 1),
                     'delta':     delta})

delta_df = pd.DataFrame(rows)
delta_df.to_csv(OUT_DIR / f'delta_factors_{HORIZONTE}.csv', index=False)
print(delta_df.pivot(index='T_years', columns='escenario', values='delta').round(3))
print(f'\nFactores delta guardados en {OUT_DIR}/delta_factors_{HORIZONTE}.csv')



Factores delta P_CC/P_ctrl — profundidad a 24.0h:
escenario  rcp45  rcp85
T_years                
2          3.293  1.884
5          2.901  1.730
10         2.662  1.611
25         2.387  1.459
50         2.207  1.356
100        2.047  1.262
500        1.739  1.076

Factores delta guardados en /workspace/data/pilot_cases/los_corrales_buelna/processed/hms_events/delta_factors_2041_2070.csv


---
## 3. HMS model structure — subbasin reading

HEC-HMS represents the Besaya basin as a set of **subbasins** (each with its own 
contributing area, CN, time of concentration) connected by channel reaches with 
routing parameters (Muskingum).

Key parameters read from the `.basin` file:

| Parameter | Symbol | Role | Typical range (Besaya) |
|-----------|--------|------|----------------------|
| Area | A (km²) | Controls total runoff volume | 10–200 km² per subbasin |
| Curve Number | CN | Runoff fraction (SCS-CN method) | 60–85 (mixed land use) |
| Time of concentration | Tc (h) | Time from peak rain to peak flow | 2–10 h |
| Clark storage coefficient | R | Smooths the UH shape | 4–15 h |

> **Calibration note:** CN and Tc are the most sensitive parameters. Without 
> calibration against observed floods, CN = 75 (average) and Tc from Témez 
> formula are reasonable first estimates, but can produce peak discharges 2–5× 
> the observed value. See the calibration warning in Section 6 and Notebook 05.


In [6]:
subbasins = (
    read_subbasin(PATH_MODEL, f'{NAME_BASIN}.basin')
    if (HMS_DIR / f'{NAME_BASIN}.basin').exists()
    else []
)
print(f'Subcuencas en Modelo_Besaya: {len(subbasins)}')
print(subbasins[:10], '...')

Subcuencas en Modelo_Besaya: 53
['W1050', 'W1030', 'W1060', 'W1000', 'W1020', 'W1010', 'W1040', 'W990', 'W950', 'W970'] ...


---
## 4. HMS file generation — alternating block design

For each return period T, a `.gage` and `.met` file pair is written with the 
alternating-block hyetograph. The time step is 1 hour, storm duration is 24 hours.

The DSS (Data Storage System) format is used for the precipitation input — 
`hecdss` writes directly to `.dss` without requiring HEC-HMS to be running. 
The output DSS contains one time series record per subbasin per return period.

**File structure:**
```
Proyecto_Hec_HMS/
├── Modelo_Besaya.basin       # subbasin parameters
├── Modelo_Besaya.met         # meteorological model (uses .gage references)
├── Modelo_Besaya.dss         # precipitation input + results output
└── Control_Design_T{n}.control  # simulation time window
```


In [7]:
# Simulation control (48h to cover storm + recession)
DESIGN_CTRL = 'Control_Design'

generate_control(
    name_model=NAME_MODEL,
    path_model=PATH_MODEL,
    name_control=DESIGN_CTRL,
    start_time='1 January 2000, 00:00',
    end_time='3 January 2000, 00:00',
    time_interval=str(TIME_STEP),
)

# Remove stale design DSS to ensure hecdss creates a fresh v7 file
_design_dss = HMS_DIR / FILE_DSS_DESIGN
if _design_dss.exists():
    _design_dss.unlink()

run_names = []
first_gage = True
HMS_INPUTS_READY = True
HMS_INPUTS_MESSAGE = 'HEC-HMS design inputs generated'

for T in RETURN_PERIODS:
    h = hietogramas[T]

    gage_name = f'Gage_T{T}'
    met_name  = f'Met_T{T}'
    run_name  = f'Run_T{T}'

    try:
        # One gage per return period — all subbasins receive the same spatially uniform precip
        generate_gage(
            name_model    = NAME_MODEL,
            names_stations= [gage_name],
            time_interval = EPART,            # DSS E-part: '1HOUR' for 60-min timestep
            path_model    = PATH_MODEL,
            start_time    = '1 January 2000, 00:00',
            end_time      = '3 January 2000, 00:00',
            file_dss      = FILE_DSS_DESIGN,  # separate v7 DSS (avoids Project_1.dss v6)
            exists_gage   = not first_gage,
        )
        first_gage = False

        fill_gage_series(
            name_station = gage_name,
            values       = h.values,
            start_time   = '1 January 2000, 00:00',
            time_interval= TIME_STEP,         # integer minutes
            path_model   = PATH_MODEL,
            file_dss     = FILE_DSS_DESIGN,
        )

        generate_met(
            name_met    = met_name,
            names_sbasin= subbasins,
            names_gage  = [gage_name] * len(subbasins),
            path_model  = PATH_MODEL,
            name_basin  = NAME_BASIN,
        )

        generate_run(
            path_model   = PATH_MODEL,
            name_model   = NAME_MODEL,
            name_run     = run_name,
            name_met     = met_name,
            name_basin   = NAME_BASIN,
            name_control = DESIGN_CTRL,
        )
        run_names.append(run_name)

    except ImportError as exc:
        HMS_INPUTS_READY = False
        HMS_INPUTS_MESSAGE = f'HEC-HMS DSS/gage generation skipped: {exc}'
        print(f'⚠ {HMS_INPUTS_MESSAGE}')
        print('  The notebook will continue with the analytical SCS-CN fallback.')
        run_names = []
        break

if run_names:
    generate_py(PATH_MODEL, NAME_MODEL, run_names)
    print(f'Generados {len(run_names)} runs: {run_names}')
else:
    print('No se generaron runs HMS ejecutables en este entorno.')


✓ Control_Design.control created and registered in Project_1.hms.
✓ Project_1.gage written (1 gages).
  ✓ Gage_T2 written to DSS.
✓ Met_T2.met written.
✓ Run 'Run_T2' added to Project_1.run.
✓ Project_1.gage written (1 gages).
  ✓ Gage_T5 written to DSS.
✓ Met_T5.met written.
✓ Run 'Run_T5' added to Project_1.run.
✓ Project_1.gage written (1 gages).
  ✓ Gage_T10 written to DSS.
✓ Met_T10.met written.
✓ Run 'Run_T10' added to Project_1.run.
✓ Project_1.gage written (1 gages).
  ✓ Gage_T25 written to DSS.
✓ Met_T25.met written.
✓ Run 'Run_T25' added to Project_1.run.
✓ Project_1.gage written (1 gages).
  ✓ Gage_T50 written to DSS.
✓ Met_T50.met written.
✓ Run 'Run_T50' added to Project_1.run.
✓ Project_1.gage written (1 gages).
  ✓ Gage_T100 written to DSS.
✓ Met_T100.met written.
✓ Run 'Run_T100' added to Project_1.run.
✓ Project_1.gage written (1 gages).
  ✓ Gage_T500 written to DSS.
✓ Met_T500.met written.
✓ Run 'Run_T500' added to Project_1.run.
✓ compute_current.py written to /works

---
## 5. HEC-HMS execution

HEC-HMS is launched as an external process using the `Hec-Hms.sh` (Linux) or 
`HecHms.exe` (Windows) binary. The Java AWT subsystem requires a display (X server) 
to initialise even in headless mode — this can block for several minutes in 
containerised environments.

**Environment variable controls:**
- `HYDRA_RUN_HEC_HMS=1` — enables execution; leave unset in Azure notebooks
- `RUN_HEC_HMS` is `False` by default → the notebook reads precomputed DSS output

When execution is disabled, the notebook falls back to an analytical SCS-CN + 
Clark UH estimate (simplified, uncalibrated). The fallback is clearly labelled 
in the output so downstream notebooks do not confuse it with calibrated results.


In [8]:
import os

RUN_HEC_HMS = os.environ.get('HYDRA_RUN_HEC_HMS', '').strip().lower() in {'1', 'true', 'yes'}
HMS_RUN_OK = False
HMS_RUN_MESSAGE = 'HEC-HMS execution not requested'

if not RUN_HEC_HMS:
    print('HEC-HMS execution skipped by default in Azure/Jupyter sessions.')
    print('Set HYDRA_RUN_HEC_HMS=1 in a validated container to run the external HMS binary.')
elif not HEC_HMS_BIN.exists():
    HMS_RUN_MESSAGE = 'HEC-HMS binary not available'
    print('HEC-HMS no disponible en esta maquina.')
    print('Opciones:')
    print('  · Docker/Azure: mount /workspace/data/hms/HEC-HMS-4.13')
    print('  · Windows: instalar HEC-HMS 4.x y ajustar HEC_HMS_BIN')
    print('  · Los hidrogramas pre-calculados estan en PROC_DIR/hms_events/')
else:
    timeout_s = int(os.environ.get('HEC_HMS_RUN_TIMEOUT', '180'))
    return_codes = []
    for run_name in run_names:
        ret = run_hms_script(
            PATH_MODEL,
            NAME_MODEL,
            [run_name],
            hms_dir=str(HEC_HMS_BIN.parent),
            timeout=timeout_s,
        )
        return_codes.append(ret)
        print(f'  {run_name}: codigo de retorno = {ret}')
        if ret != 0:
            break
    HMS_RUN_OK = bool(return_codes) and all(ret == 0 for ret in return_codes)
    HMS_RUN_MESSAGE = 'HEC-HMS completed' if HMS_RUN_OK else f'HEC-HMS failed/stopped: {return_codes}'
    print(HMS_RUN_MESSAGE)


HEC-HMS execution skipped by default in Azure/Jupyter sessions.
Set HYDRA_RUN_HEC_HMS=1 in a validated container to run the external HMS binary.


---
## 6. Hydrograph reading and results

The preferred result source is the HEC-HMS DSS output. If the DSS cannot be read
or HEC-HMS fails in the container, the notebook reports a simplified analytical
fallback. The source is printed explicitly so downstream notebooks do not confuse
fallback values with calibrated model simulations.

**Hydrograph comparison — Control vs CC scenarios:**

The three-panel comparison (Control, RCP4.5, RCP8.5) for T=100 shows:
- The **shape** of the hydrograph (rising limb slope, recession) is unchanged — 
  the CC scenarios only scale the precipitation depth via delta factors
- The **peak** increases by the CC amplification factor (typically +5–20% for 
  RCP4.5, +10–30% for RCP8.5 in Cantabria for the 2041–2070 horizon)
- The **volume** increases proportionally (same Tc, same CN, higher rain)


In [9]:
from pyhydra.modeling.hydrology.hec_hms import read_dss6_timeseries

dss_path = HMS_DIR / f'{NAME_MODEL}.dss'

peak_flows = {}
hydrographs = {}

if HMS_RUN_OK and dss_path.exists() and run_names:
    try:
        for T, run_name in zip(RETURN_PERIODS, run_names):
            # pathname_prefix identifies the series; monthly date suffix found automatically.
            # read_dss6_timeseries returns a DataFrame with columns 'datetime' and 'value'.
            df_dss = read_dss6_timeseries(
                str(dss_path),
                pathname_prefix='//OUTLET/FLOW',
            )
            ts = df_dss['value'].rename(f'T{T}')
            hydrographs[T] = ts
            peak_flows[T]  = float(ts.max())
        if peak_flows:
            result_source = 'HEC-HMS DSS'
            print('Hidrogramas leídos desde DSS ✓')
        else:
            result_source = 'SCS-CN fallback (not calibrated HEC-HMS)'
            print('DSS disponible, pero no hay runs HMS leíbles en este entorno.')
    except Exception as e:
        print(f'No se pudo leer DSS: {e}')
        result_source = 'SCS-CN fallback (not calibrated HEC-HMS)'
        print('→ Usando simulación SCS-CN simplificada como alternativa; resultados NO calibrados')

if not HMS_RUN_OK:
    print(f'HEC-HMS output not used: {HMS_RUN_MESSAGE}')

# Fallback: simplified SCS-CN simulation (no HEC-HMS required)
if not peak_flows:
    result_source = 'SCS-CN fallback (not calibrated HEC-HMS)'
    print(f'\nSimulación SCS-CN simplificada (Clark UH) — {result_source}:')

    def scs_cn_runoff(P_mm, CN=75):
        """Abstracciones SCS: devuelve escorrentía directa (mm)."""
        S = 25400 / CN - 254
        Ia = 0.2 * S
        Q = np.where(P_mm > Ia, (P_mm - Ia)**2 / (P_mm - Ia + S), 0)
        return Q

    BASIN_AREA_KM2 = 288.0   # área total de la cuenca del Besaya hasta Torrelavega
    CN_BESAYA      = 75       # número de curva medio (suelos B, cultivos)
    Tc_H           = 6.0      # tiempo de concentración (h)

    for T in RETURN_PERIODS:
        h = hietogramas[T]
        runoff_mm = scs_cn_runoff(h.cumsum().values, CN=CN_BESAYA)
        incremental_runoff = np.diff(np.concatenate([[0], runoff_mm]))

        # Clark UH simplificado: triangular con tp=0.6*Tc, tb=2.67*tp
        tp = 0.6 * Tc_H
        peak_uh = 2.08 * BASIN_AREA_KM2 / tp   # m³/s por mm

        # Hyetograph-UH convolution
        n_uh = int(2.67 * tp / (TIME_STEP / 60)) + 1
        uh = np.zeros(n_uh)
        n_rise = int(tp / (TIME_STEP / 60))
        uh[:n_rise+1] = np.linspace(0, peak_uh, n_rise+1)
        uh[n_rise:]   = np.linspace(peak_uh, 0, n_uh - n_rise)

        Q = np.convolve(incremental_runoff, uh)[:len(h)]
        ts = pd.Series(Q, index=h.index, name=f'T{T}')

        hydrographs[T] = ts
        peak_flows[T]  = float(ts.max())
        print(f'  T={T:>4} años: Qp = {peak_flows[T]:.1f} m³/s')


HEC-HMS output not used: HEC-HMS execution not requested

Simulación SCS-CN simplificada (Clark UH) — SCS-CN fallback (not calibrated HEC-HMS):
  T=   2 años: Qp = 750.7 m³/s
  T=   5 años: Qp = 1259.5 m³/s
  T=  10 años: Qp = 1655.7 m³/s
  T=  25 años: Qp = 2241.8 m³/s
  T=  50 años: Qp = 2724.4 m³/s
  T= 100 años: Qp = 3241.1 m³/s
  T= 500 años: Qp = 4552.1 m³/s


In [10]:
# Exportar hidrogramas
hydro_df = pd.DataFrame(hydrographs)
hydro_df.columns = [f'T{T}_m3s' for T in hydro_df.columns]
hydro_df.to_csv(OUT_DIR / 'hidrogramas_disenio.csv')

# Tabla de caudales pico
pq_df = pd.Series(peak_flows, name='Qpico_m3s').rename_axis('T_years').reset_index()
pq_df.to_csv(OUT_DIR / 'caudales_pico_disenio.csv', index=False)

print('Caudales pico:')
print(pq_df.to_string(index=False))

Caudales pico:
 T_years   Qpico_m3s
       2  750.721263
       5 1259.484143
      10 1655.660986
      25 2241.772219
      50 2724.387176
     100 3241.139495
     500 4552.096242


---
## ⚠ Warning: uncalibrated peak flow values

The peak flows shown above come from the **simplified SCS-CN + Clark UH simulation**, 
which activates when HEC-HMS is unavailable in the execution environment.  
**These values are NOT calibrated against observed Besaya streamflow** and must not 
be used as final design discharges.

**Calibration gap:**  
The largest observed discharge at Torrelavega (station 1237) over 2000–2008 is 
**~285 m³/s**. The uncalibrated simulation (CN = 75, A = 288 km², Tc = 6 h) 
produces T100 ≈ 3200 m³/s — an order of magnitude too high. A calibration against 
observed events would substantially reduce CN or increase Tc.

**Design flows for HEC-RAS (Notebooks 07 and 08):**  
The **hybrid methodology** (Met-Hab, Notebook 06) provides physically calibrated 
design hydrographs derived from observed streamflow events. These are the flows 
used for flood inundation mapping and should be preferred over the classical 
SCS-CN estimates from this notebook.

**When is the classical methodology appropriate?**
- For preliminary screening before calibration data are available
- For ungauged sub-catchments where no observed hydrograph exists
- For estimating the shape (not magnitude) of the design hyetograph


In [11]:
# Design hydrograph plot
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=False)
axes = axes.ravel()
colors = plt.cm.plasma(np.linspace(0, 0.85, len(RETURN_PERIODS)))

for ax, (T, c) in zip(axes, zip(RETURN_PERIODS, colors)):
    h = hydrographs[T]
    ax.plot(range(len(h)), h.values, color=c, lw=1.5)
    ax.fill_between(range(len(h)), h.values, alpha=0.3, color=c)
    ax.set(title=f'T = {T} años  (Qp={peak_flows[T]:.0f} m³/s)',
           xlabel='Paso (h)', ylabel='Q (m³/s)')

axes[-1].set_visible(False)
plt.suptitle('Hidrogramas de diseño — Cuenca del Besaya', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'hidrogramas_disenio.png', dpi=150)
plt.show()

In [12]:
# Curva Q-T (caudales pico vs periodo de retorno)
T_vals = list(peak_flows.keys())
Q_vals = list(peak_flows.values())

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(T_vals, Q_vals, 'o-', color='steelblue', lw=2, ms=8)
for T, Q in zip(T_vals, Q_vals):
    ax.annotate(f'{Q:.0f}', (T, Q), xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=9)
ax.set(xlabel='Periodo de retorno T (años)', ylabel='Caudal pico (m³/s)',
       title='Curva Q-T — Cuenca del Besaya (metodología clásica)')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'curva_QT.png', dpi=150)
plt.show()